In [15]:
import pandas as pd
import json
from pathlib import Path
import re

In [16]:
def book_id_to_path(id, book_id_to_book, csv_data_directory):
    book_info = book_id_to_book[id]
    start_year = book_info["year_start"]
    end_year = book_info["year_end"]
    source = book_info["source"].lower()

    return Path(csv_data_directory).glob(
        f"*{book_info["parish_normalized"]}/*{start_year}-{end_year}_{source}"
    )

In [17]:
csv_data_directory = "data/migration-data-csv-release-v1"
book_ids_df = pd.read_json("outputs/json/book_ids.json", orient="index")
book_ids_df.columns = ["parish_normalized", "year_start", "year_end", "source"]
book_ids_df = book_ids_df.rename_axis("book_id")
book_ids_to_info = book_ids_df.to_dict(orient="index")

id_to_book = book_ids_df.to_dict(orient="index")
book_path_stems = tuple(tuple(i) for i in book_ids_df.index.map(
    lambda x: book_id_to_path(x, id_to_book, csv_data_directory)
).to_list())
book_path_stems = pd.DataFrame({"stem": tuple(i[0].stem if len(i) > 0 else None for i in book_path_stems)}, index=book_ids_df.index)

book_to_id = {
    v: k
    for k, v in (
        book_ids_df["parish_normalized"]
        + "_" + book_path_stems["stem"]
    )
    .to_dict()
    .items()
}

In [18]:
original_book_metadata = pd.read_csv("data/csv/Moving_record_parishes_with_formats_v2.csv")
original_book_metadata = original_book_metadata.dropna(subset="url")
original_book_metadata["book_id"] = original_book_metadata["url"].apply(lambda x: int(x.split("=")[-1]))
original_book_metadata = original_book_metadata.drop("url", axis="columns")

In [19]:
event_year_df = pd.read_csv("outputs/csv/year_predictions_extended.csv")
book_migration_metadata = pd.read_csv("outputs/csv/book_migration_direction_info.csv")
parish_df = pd.read_csv("data/csv/parish.csv")

In [20]:
print_columns = pd.read_csv("data/csv/Moving_record_formats_v2.csv")
print_columns["side"] = print_columns["sarakkeiden lkm"].apply(
    lambda x: re.sub(r"[\d\s]", "", x)
)
df_cols = list(print_columns.columns[6:])
df_cols.insert(0, df_cols.pop(-1))
df_cols.insert(0, "print type")
print_columns = print_columns[df_cols]

printed_cols = (
    print_columns[df_cols[2:]]
    .astype(str)
    .to_records(index=False)
    .tolist()
)
for i, p_type_cols in enumerate(printed_cols):
    corrected = list(filter(lambda x: not pd.isna(x), p_type_cols))
    printed_cols[i] = corrected

print(printed_cols)

print_columns["sarakkeet"] = printed_cols
print_columns = print_columns.drop(df_cols[3:], axis="columns")

[['kuukausi', 'päivä', 'nimi ja sääty', 'miespuoli', 'naispuoli', '(syntymä)päivä', '(syntymä)paikka', 'empty', 'leski, nainut, naimaton', 'elatuskeino', '(muuttokirjan)päivä', '(muuttokirjan)paikka', 'lehti kirkonkirjasta', 'muistutuksia'], ['kuukausi', 'päivä', 'muuttokirjan numero', 'nimi ja sääty', 'miespuoli', 'naispuoli', '(syntymä)päivä', '(syntymä)paikka', 'empty', 'leski, nainut, naimaton', 'elatuskeino', 'muuttopaikka', 'lehti kirkonkirjasta', 'muistutuksia'], ['juokseva numero', 'muuttokirja sisäänannettiin(kuukausi)', 'muuttokirja sisäänannettiin(päivä)', 'merkitty pääkirjaan sivulle', 'sisäänmuuttaneitten sääty, nimi ja ammatti', 'mistä seurakunnasta muutettiin', 'muuttokirjan päivämäärä', 'numero', 'miespuolta', 'vaimopuolta', 'ilmoitusvastaanottamisesta'], ['juokseva numero', 'muuttokirja sisäänannettiin(kuukausi)', 'muuttokirja sisäänannettiin(päivä)', 'merkitty pääkirjaan sivulle', 'sisäänmuuttaneitten sääty, nimi ja ammatti', 'mistä seurakunnasta muutettiin', 'muuttok

In [21]:
print_columns.to_csv("outputs/csv/print_type_columns.csv", index=False)

In [22]:
cols = print_columns["sarakkeet"].to_list()
cols

[['kuukausi',
  'päivä',
  'nimi ja sääty',
  'miespuoli',
  'naispuoli',
  '(syntymä)päivä',
  '(syntymä)paikka',
  'empty',
  'leski, nainut, naimaton',
  'elatuskeino',
  '(muuttokirjan)päivä',
  '(muuttokirjan)paikka',
  'lehti kirkonkirjasta',
  'muistutuksia'],
 ['kuukausi',
  'päivä',
  'muuttokirjan numero',
  'nimi ja sääty',
  'miespuoli',
  'naispuoli',
  '(syntymä)päivä',
  '(syntymä)paikka',
  'empty',
  'leski, nainut, naimaton',
  'elatuskeino',
  'muuttopaikka',
  'lehti kirkonkirjasta',
  'muistutuksia'],
 ['juokseva numero',
  'muuttokirja sisäänannettiin(kuukausi)',
  'muuttokirja sisäänannettiin(päivä)',
  'merkitty pääkirjaan sivulle',
  'sisäänmuuttaneitten sääty, nimi ja ammatti',
  'mistä seurakunnasta muutettiin',
  'muuttokirjan päivämäärä',
  'numero',
  'miespuolta',
  'vaimopuolta',
  'ilmoitusvastaanottamisesta'],
 ['juokseva numero',
  'muuttokirja sisäänannettiin(kuukausi)',
  'muuttokirja sisäänannettiin(päivä)',
  'merkitty pääkirjaan sivulle',
  'sisä

In [23]:
all_column_names = []
for i in range(len(cols)):
    all_column_names += cols[i]

In [24]:
print(
    len(all_column_names),
    len(set(all_column_names))
)

923 367


In [25]:
all_column_names = sorted(set(all_column_names))
all_column_names

['(antal personer)mankön',
 '(antal personer)qvinkön',
 '(asev.)ilm',
 '(asev.)ilm.',
 '(asev.)synt. vuosi',
 '(attestets beskafferhet)bokläsning',
 '(attestets beskafferhet)frägd',
 '(attestets beskafferhet)nattvardsgång',
 '(attestets beskafferhet)äktenskap',
 '(begått h. h. nattvard)18',
 '(de utflyttades antal)finska stads församling',
 '(de utflyttades antal)landsförsamling',
 '(de utflyttades antal)svenska församling',
 '(flyttningsbetyget datum)dag',
 '(flyttningsbetyget datum)månad',
 '(flyttningsbetyget inlämnades)dag',
 '(flyttningsbetyget inlämnades)månad',
 '(flyttningsbetygets)datum',
 '(flyttningsbetygets)nummer',
 '(födelse)dag',
 '(födelse)ort',
 '(födelse)år',
 '(födelse)år och datum',
 '(henkiluku)miehenpuolta',
 '(henkiluku)vaimonpuolta',
 '(henkilöluku)miehenpuolta',
 '(henkilöluku)mp',
 '(henkilöluku)mp.',
 '(henkilöluku)np',
 '(henkilöluku)np.',
 '(henkilöluku)vaimonpuolta',
 '(henkilöluku)vp',
 '(ihmisluku)miespuolta',
 '(ihmisluku)naispuolta',
 '(käynti herran e

Manual review and categorization of column names done at this point.

Results:

In [26]:
with open("outputs/json/print_type_column_names_categorized.json", "r") as fp:
    translate_column_name = json.load(fp)

for print_cols in cols:
    for x in print_cols:
        print(x, "~", translate_column_name[x])

    print("---")

kuukausi ~ other
päivä ~ other
nimi ja sääty ~ name
miespuoli ~ person_count
naispuoli ~ person_count
(syntymä)päivä ~ other
(syntymä)paikka ~ other
empty ~ 
leski, nainut, naimaton ~ other
elatuskeino ~ 
(muuttokirjan)päivä ~ other
(muuttokirjan)paikka ~ place
lehti kirkonkirjasta ~ other
muistutuksia ~ other
---
kuukausi ~ other
päivä ~ other
muuttokirjan numero ~ other
nimi ja sääty ~ name
miespuoli ~ person_count
naispuoli ~ person_count
(syntymä)päivä ~ other
(syntymä)paikka ~ other
empty ~ 
leski, nainut, naimaton ~ other
elatuskeino ~ 
muuttopaikka ~ place
lehti kirkonkirjasta ~ other
muistutuksia ~ other
---
juokseva numero ~ other
muuttokirja sisäänannettiin(kuukausi) ~ other
muuttokirja sisäänannettiin(päivä) ~ other
merkitty pääkirjaan sivulle ~ other
sisäänmuuttaneitten sääty, nimi ja ammatti ~ name
mistä seurakunnasta muutettiin ~ source
muuttokirjan päivämäärä ~ other
numero ~ other
miespuolta ~ person_count
vaimopuolta ~ person_count
ilmoitusvastaanottamisesta ~ other
--